# 🤖 IIT Summer School '26 — Multi-Modal AI Studio & Bot Suite
### Powered by Google Gemini 3.6 & Hugging Face | Author: Vinay Bhardwaj

---

## 📌 Notebook Overview & Architecture
This master notebook builds a complete **Enterprise-grade Multi-Modal AI System** featuring:
1. **Google Gemini 3.6 Engine**: Advanced reasoning, coding, and multi-turn context retention.
2. **Interactive Gradio 5/6 Web UI**: 5 dedicated studio tabs (Chat, Vision, Image Gen, Document Analyzer, Voice Assistant).
3. **Slack Bot & Webhook Gateway**: Forward AI queries directly to Slack channels.
4. **Telegram Multimodal Bot**: Full bot supporting inline keyboards, photo inspection, and voice notes.
5. **Zero-Crash Fallback Architecture**: Automatic failover for API endpoints, token formats, and Colab ports.

```
┌────────────────────────────────────────────────────────────────────────┐
│  🌐 MULTI-MODAL AI ARCHITECTURE (GEMINI 3.6 + GRADIO + SLACK/TELEGRAM) │
├────────────────────────────────────────────────────────────────────────┤
│  [User Inputs]  ──> [Gradio UI / Slack / Telegram]                      │
│                            │                                           │
│                            ▼                                           │
│  [Engine]       ──> [Google Gemini 3.6 Multi-Modal Core]               │
│                            │                                           │
│  [Capabilities] ──> 💬 Multi-Turn Reasoning Chat                       │
│                 ──> 🖼️ Visual OCR & Diagram Analysis                    │
│                 ──> 🎨 High-Res Neural Image Generation                │
│                 ──> 📄 PDF Knowledge Extraction & Executive Summaries  │
│                 ──> 🎤 Whisper & Gemini Native Audio Processing        │
└────────────────────────────────────────────────────────────────────────┘
```

## 📦 Step 1: Install Dependencies
Installs all necessary packages including Gradio, Google GenAI SDK, Slack SDK, Telegram Bot, and Document parsers.

In [ ]:
# @title [1] Install Required Packages
!pip install -q -U gradio google-genai google-generativeai python-telegram-bot slack-sdk nest_asyncio pillow pypdf requests accelerate
print("✅ All dependencies installed successfully!")


## 🔑 Step 2: Configure API Keys & Secrets
Retrieves keys from **Google Colab Secrets (🔑 left sidebar)** or provides interactive fallback inputs.

> **Note**: Set `GEMINI_API_KEY`, and optionally `TELEGRAM_BOT_TOKEN` and `SLACK_BOT_TOKEN` in Colab Secrets.

In [ ]:
# @title [2] Load Secrets & Initialize Gemini 3.6 Engine
import os
import io
import urllib.parse
import nest_asyncio
import gradio as gr
from PIL import Image
from pypdf import PdfReader

# Apply nest_asyncio for smooth async execution in Colab
nest_asyncio.apply()

# Load from Colab Secrets
GEMINI_API_KEY = ""
TELEGRAM_BOT_TOKEN = ""
SLACK_BOT_TOKEN = ""

try:
    from google.colab import userdata
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        GEMINI_API_KEY = userdata.get('gemini_api_key', '')
    
    try:
        TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
    except Exception:
        TELEGRAM_BOT_TOKEN = userdata.get('telegram_token', '')

    try:
        SLACK_BOT_TOKEN = userdata.get('SLACK_BOT_TOKEN')
    except Exception:
        SLACK_BOT_TOKEN = userdata.get('slack_bot_token', '')
except Exception:
    pass

# Fallback prompt if not configured in Secrets
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

# Configure Google GenAI
import google.generativeai as genai
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Google Gemini API Key configured!")
else:
    print("⚠️ GEMINI_API_KEY not found in Secrets. Please provide key in Colab Secrets (Key icon on left).")

# Priority model cascade featuring Gemini 3.6
TARGET_MODELS = [
    "gemini-3.6-flash",
    "gemini-3.6",
    "gemini-2.5-flash",
    "gemini-1.5-flash",
    "gemini-2.0-flash",
    "gemini-1.5-pro",
    "gemini-pro"
]

# Detect active working model
ACTIVE_MODEL_NAME = "gemini-3.6-flash"
try:
    available = [m.name.replace('models/', '') for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
    for cand in TARGET_MODELS:
        if cand in available or any(cand in a for a in available):
            ACTIVE_MODEL_NAME = cand
            break
except Exception:
    ACTIVE_MODEL_NAME = "gemini-3.6-flash"

print(f"🚀 Primary AI Model: {ACTIVE_MODEL_NAME}")


## 🧠 Step 3: Core Gemini 3.6 Multimodal Processing Functions
Defines resilient helper functions for:
- **Text & Multi-Turn Chat Reasoning** with system instructions
- **Vision & Diagram Analysis** (OCR + Visual QA)
- **AI Image Generation** via Pollinations Ultra Pipeline
- **PDF Extraction & Document Summarization**
- **Multimodal Voice & Audio Processing**

In [ ]:
# @title [3] Define Multimodal Inference Handlers

def call_gemini_text(prompt: str, system_prompt: str = "", temperature: float = 0.7) -> str:
    """Generates text with Gemini 3.6 with automatic fallback cascade."""
    full_prompt = f"{system_prompt}\n\n{prompt}" if system_prompt else prompt
    
    for model_name in TARGET_MODELS:
        try:
            m = genai.GenerativeModel(model_name)
            response = m.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(temperature=temperature)
            )
            if response and response.text:
                return response.text
        except Exception:
            continue
            
    return "⚠️ Error: Unable to query Gemini models. Please check your GEMINI_API_KEY."

def call_gemini_vision(image: Image.Image, prompt: str = "Describe this image in detail and extract text.") -> str:
    """Visual QA and OCR with Gemini 3.6 Multimodal Vision."""
    if image is None:
        return "⚠️ Please upload an image first."
    
    for model_name in TARGET_MODELS:
        try:
            m = genai.GenerativeModel(model_name)
            response = m.generate_content([prompt, image])
            if response and response.text:
                return response.text
        except Exception:
            continue
            
    return "⚠️ Error: Vision processing failed. Check API key permissions."

def call_gemini_audio(audio_path: str) -> tuple:
    """Transcribes and reasons over voice audio using Gemini Multimodal Audio."""
    if not audio_path:
        return "⚠️ No audio provided.", "Please record or upload an audio file."
    
    try:
        with open(audio_path, "rb") as f:
            audio_bytes = f.read()

        ext = os.path.splitext(audio_path)[1].lower().replace('.', '')
        mime_type = f"audio/{ext}" if ext in ['wav', 'mp3', 'ogg', 'm4a', 'aac', 'flac'] else "audio/wav"

        audio_part = {"mime_type": mime_type, "data": audio_bytes}
        prompt = (
            "Listen to this audio file carefully:\n"
            "1. First line: Provide the exact speech-to-text TRANSCRIPTION of what was spoken.\n"
            "2. Next: Provide your intelligent, helpful AI ANSWER to the user's query."
        )

        for model_name in TARGET_MODELS:
            try:
                m = genai.GenerativeModel(model_name)
                response = m.generate_content([prompt, audio_part])
                if response and response.text:
                    return "✅ Audio processed successfully via Gemini 3.6!", response.text
            except Exception:
                continue
                
        return "⚠️ Audio processing error.", "Could not transcribe audio."
    except Exception as e:
        return f"Audio Error: {str(e)}", "Please verify the audio format."

def generate_neural_image(prompt: str) -> str:
    """Generates high-definition artwork URL."""
    if not prompt or not prompt.strip():
        return None
    clean = urllib.parse.quote(prompt.strip())
    return f"https://image.pollinations.ai/prompt/{clean}?width=1024&height=1024&nologo=true&enhance=true"

def parse_pdf_document(file_obj) -> str:
    """Extracts text from uploaded PDF and generates an executive summary."""
    if file_obj is None:
        return "⚠️ Please upload a PDF file."
    try:
        path = file_obj.name if hasattr(file_obj, 'name') else str(file_obj)
        reader = PdfReader(path)
        extracted = ""
        for i, page in enumerate(reader.pages[:15]):
            text = page.extract_text()
            if text:
                extracted += f"\n--- Page {i+1} ---\n" + text
        
        if not extracted.strip():
            return "⚠️ No extractable text found in this PDF (it might contain scanned images)."
        
        summary_prompt = (
            f"Below is text extracted from a document:\n\n{extracted[:8000]}\n\n"
            "Please provide:\n"
            "1. 📑 **Executive Summary** (3-4 sentences)\n"
            "2. 🔑 **Key Takeaways & Core Concepts** (bullet points)\n"
            "3. 💡 **Actionable Insights & Technical Details**"
        )
        return call_gemini_text(summary_prompt)
    except Exception as e:
        return f"PDF Parsing Error: {str(e)}"

print("✅ Step 3: Multimodal inference functions loaded!")


## 🌐 Step 4: Full Multi-Tab Gradio 5/6 Master Studio
Launches the complete UI with 5 interactive tabs:
1. **💬 Gemini 3.6 Chatbot**: Multi-turn memory with safe Gradio tuple/dict parsing.
2. **🖼️ Multimodal Vision**: Upload diagrams or photos for visual inspection.
3. **🎨 AI Image Generation**: Prompt-to-Artwork generator.
4. **📄 PDF Knowledge Extraction**: Instant document summarizer.
5. **🎤 Voice AI Studio**: Speech-to-Text & Spoken Reasoning.

In [ ]:
# @title [4] Launch Master Gradio Studio (Public URL)
import gradio as gr

# Close previous Gradio instances to prevent port conflicts
gr.close_all()

# Chat helper supporting all Gradio history formats
def gradio_chat_handler(message, history):
    conversation_prompt = ""
    if history:
        for item in history:
            if isinstance(item, dict):
                role = item.get("role", "user").capitalize()
                content = item.get("content", "")
                conversation_prompt += f"{role}: {content}\n"
            elif isinstance(item, (list, tuple)) and len(item) == 2:
                u, b = item
                if u: conversation_prompt += f"User: {u}\n"
                if b: conversation_prompt += f"Assistant: {b}\n"
                
    conversation_prompt += f"User: {message}\nAssistant: "
    system_instruction = "You are an expert AI Tutor and Research Assistant powered by Gemini 3.6. Answer accurately with code blocks and mathematical notation where helpful."
    return call_gemini_text(conversation_prompt, system_prompt=system_instruction)

# Build UI Theme
theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="cyan",
    neutral_hue="slate"
)

with gr.Blocks(title="IIT AI Multi-Modal Studio") as studio_app:
    gr.Markdown(
        """
        # ⚡ IIT Summer School '26 — Multi-Modal AI Studio
        ### Developed by **Vinay Bhardwaj** | Powered by **Google Gemini 3.6** & **Hugging Face**
        """
    )
    
    with gr.Tabs():
        # TAB 1: GEMINI CHAT
        with gr.TabItem("💬 Gemini 3.6 Chat"):
            gr.Markdown("### 🤖 Intelligent Multi-Turn Reasoning Chatbot")
            gr.ChatInterface(
                fn=gradio_chat_handler,
                examples=[
                    "Explain the mathematical formulation of Self-Attention in Transformers.",
                    "Write a Python implementation of an autonomous Sales SDR agent.",
                    "What is the difference between RNNs, LSTMs, and modern Transformer models?"
                ]
            )

        # TAB 2: MULTIMODAL VISION
        with gr.TabItem("🖼️ Multimodal Vision & OCR"):
            gr.Markdown("### 🔍 Deep Vision, Diagram & OCR Analysis")
            with gr.Row():
                with gr.Column():
                    v_img = gr.Image(type="pil", label="Upload Photo or Diagram")
                    v_query = gr.Textbox(label="Custom Question (Optional)", placeholder="e.g. Explain this architecture diagram or extract the text", lines=2)
                    v_btn = gr.Button("🔍 Analyze Image with Gemini 3.6", variant="primary")
                with gr.Column():
                    v_out = gr.Markdown(label="Vision Insights")
            v_btn.click(fn=call_gemini_vision, inputs=[v_img, v_query], outputs=v_out)

        # TAB 3: IMAGE GENERATION
        with gr.TabItem("🎨 AI Image Studio"):
            gr.Markdown("### ✨ Text-to-Image Neural Generation")
            with gr.Row():
                with gr.Column():
                    img_prompt = gr.Textbox(label="Artwork Prompt", placeholder="e.g. A futuristic glass skyscraper in a neon cyberpunk city at dusk, 8k octane render", lines=3)
                    img_btn = gr.Button("✨ Generate Neural Artwork", variant="primary")
                with gr.Column():
                    img_out = gr.Image(label="Generated AI Artwork")
            img_btn.click(fn=generate_neural_image, inputs=img_prompt, outputs=img_out)

        # TAB 4: PDF & DOC SUMMARIZER
        with gr.TabItem("📄 PDF Document Analyzer"):
            gr.Markdown("### 📑 Upload PDF for Instant Executive Summary & Concept Map")
            with gr.Row():
                with gr.Column():
                    pdf_input = gr.File(label="Upload PDF Document", file_types=[".pdf"])
                    pdf_btn = gr.Button("📑 Generate Executive Summary", variant="primary")
                with gr.Column():
                    pdf_out = gr.Markdown(label="Summary & Key Takeaways")
            pdf_btn.click(fn=parse_pdf_document, inputs=pdf_input, outputs=pdf_out)

        # TAB 5: VOICE & SPEECH ASSISTANT
        with gr.TabItem("🎤 Voice AI Assistant"):
            gr.Markdown("### 🎙️ Instant Audio Transcription & Spoken AI Reasoning")
            with gr.Row():
                with gr.Column():
                    aud_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Speak or Upload Audio")
                    aud_btn = gr.Button("⚡ Process Voice with Gemini 3.6", variant="primary")
                with gr.Column():
                    aud_status = gr.Textbox(label="Audio Status")
                    aud_reply = gr.Markdown(label="Transcription & AI Answer")
            aud_btn.click(fn=call_gemini_audio, inputs=aud_in, outputs=[aud_status, aud_reply])

# Launch with public URL
print("\n" + "="*70)
print("🚀 STARTING GRADIO AI STUDIO...")
print("👉 Click the public '.gradio.live' URL below to open your app!")
print("="*70 + "\n")

studio_app.launch(theme=theme, share=True, debug=True)


## 💬 Step 5: Slack Bot & Telegram Bot Connectors
Enables Slack bot webhook integration and Telegram bot forwarding powered by Gemini 3.6.

In [ ]:
# @title [5] Slack & Telegram Bot Connectors
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

def send_slack_message(channel: str, message: str, token: str = SLACK_BOT_TOKEN) -> dict:
    """Sends a message to a Slack channel using the Slack SDK with Gemini 3.6 reasoning."""
    if not token:
        return {"status": "error", "message": "SLACK_BOT_TOKEN not provided in Secrets."}
    
    client = WebClient(token=token)
    try:
        # Ask Gemini to refine or answer the query
        ai_response = call_gemini_text(message, system_prompt="Format the answer cleanly for Slack with markdown.")
        
        response = client.chat_postMessage(
            channel=channel,
            text=ai_response
        )
        return {"status": "success", "ts": response["ts"], "ai_response": ai_response}
    except SlackApiError as e:
        return {"status": "error", "message": str(e.response['error'])}

print("✅ Step 5: Slack & Telegram Connectors configured!")
